# 🤖 Orchestrator-Worker으로 AI 팀 만들어보기 (Supervisor & Sub-agent 소통 패턴)

이번 실습에서는 **Supervisor 패턴**을 활용하여 중앙의 Supervisor 에이전트가
전문화된 Sub-agent들을 **도구(tool)로 호출**하고, Sub-agent가 결과를 Supervisor에게 보고하는
**양방향 소통 구조**를 구현합니다.

> 📢 **기존 Orchestrator-Worker vs Supervisor 패턴**
> 
> | | 기존 (Send API) | Supervisor 패턴 |
> |---|---|---|
> | **Worker 타입** | 단순 LLM 호출 | 자체 도구를 가진 Sub-agent |
> | **소통 방식** | 일방적 작업 배분 → 결과 수집 | Sub-agent ↔ Supervisor 양방향 소통 |
> | **작업 위임** | `Send()`로 정적 배분 | Supervisor가 tool call로 **동적 위임** |
> | **컨텍스트 전달** | 고정된 payload | `ToolRuntime`으로 Supervisor 전체 맥락 전달 |
> | **사람 검토** | 없음 | `HumanInTheLoopMiddleware`로 검토 가능 |

### 아키텍처: 3단 계층 구조 (3 Sub-agent)

```
┌──────────────────────────────────────────────────────────┐
│  Supervisor Agent (최상위 - 작업 위임)                     │  ← 사용자와 소통
│  tools: [research_topic, write_section, run_code]        │
└──────────┬──────────────────┬──────────────────┬─────────┘
           │                  │                  │
  ┌────────▼────────┐ ┌──────▼────────┐ ┌──────▼────────┐
  │ Research Agent   │ │ Writer Agent  │ │ Code Agent    │  ← 전문 Sub-agent
  │ tools: [EXA     │ │ tools:        │ │ tools: [E2B   │
  │  search_web,    │ │ [validate_    │ │  code_inter-  │
  │  search_news]   │ │  markdown]    │ │  preter]      │
  └─────────────────┘ └───────────────┘ └───────────────┘
```

### 🎯 학습 목표
1. `create_agent()`로 전문 Sub-agent 생성하기
2. Sub-agent를 **도구(tool)**로 래핑하여 Supervisor에 제공하기
3. **EXA API**로 실시간 웹 검색 통합하기
4. **E2B 샌드박스**로 안전한 코드 실행 통합하기
5. (Advanced) `ToolRuntime`으로 Supervisor 컨텍스트를 Sub-agent에 전달하기
6. (Advanced) `HumanInTheLoopMiddleware`로 사람의 검토 추가하기

In [ ]:
%pip install -qU langchain langgraph langchain-openai python-dotenv exa_py e2b-code-interpreter

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# API 키 설정
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "your-openai-api-key"

# EXA API - https://exa.ai 에서 무료 발급
if not os.getenv("EXA_API_KEY"):
    os.environ["EXA_API_KEY"] = "your-exa-api-key"

# E2B API - https://e2b.dev/docs 에서 무료 발급
if not os.getenv("E2B_API_KEY"):
    os.environ["E2B_API_KEY"] = "your-e2b-api-key"

print("✅ 환경 설정 완료!")
print("   - OPENAI_API_KEY: LLM 호출")
print("   - EXA_API_KEY: 웹 검색 (https://exa.ai)")
print("   - E2B_API_KEY: 샌드박스 코드 실행 (https://e2b.dev)")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# 환경 변수 확인
print("=== LangSmith 설정 상태 ===")
print(f"LANGSMITH_API_KEY: {'설정됨' if os.getenv('LANGSMITH_API_KEY') else '미설정 ⚠️'}")
print(f"LANGSMITH_TRACING: {os.getenv('LANGSMITH_TRACING', '미설정')}")
print(f"LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT', '미설정 (default 사용)')}")
print(f"OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정 ⚠️'}")
# Tracing 활성화 (환경 변수로 설정하는 것이 권장됨)
os.environ["LANGSMITH_TRACING"] = "true"

# 프로젝트 이름 설정 (선택)
os.environ["LANGSMITH_PROJECT"] = "fc-agent-bible-appendix"

print("Tracing이 활성화되었습니다.")

---
## 1. Low-level 도구(Tool) 정의

이전 실습에서는 Mock 데이터를 사용했지만, 이번에는 **실제 API**를 사용합니다:

| 도구 | API | 역할 |
|------|-----|------|
| `search_web` | **EXA** | 웹 검색 (논문, 기사, 문서) |
| `search_news` | **EXA** | 최신 뉴스 검색 (날짜 필터) |
| `validate_markdown_section` | 로컬 | 마크다운 품질 검증 |
| `e2b_code_interpreter` | **E2B** | 격리된 샌드박스에서 Python 코드 실행 |

**핵심 포인트**: Sub-agent가 직접 사용하는 **low-level 도구**들입니다. Supervisor는 이 도구들을 직접 보지 않습니다.

In [ ]:
from exa_py import Exa
from langchain.tools import tool
from datetime import datetime, timedelta

# EXA 클라이언트 초기화
exa_client = Exa(api_key=os.environ.get("EXA_API_KEY"))


@tool
def search_web(query: str, num_results: int = 5) -> str:
    """EXA API를 사용하여 웹 검색을 수행합니다.

    최신 연구 동향, 기술 정보, 논문, 문서 등을 검색할 때 사용하세요.

    Args:
        query: 검색할 쿼리
        num_results: 반환할 결과 수 (기본값: 5)
    """
    try:
        results = exa_client.search_and_contents(
            query=query,
            type="auto",
            num_results=num_results,
            text={"max_characters": 2000},
        )

        if not results.results:
            return f"'{query}'에 대한 검색 결과가 없습니다."

        output = []
        for i, result in enumerate(results.results, 1):
            output.append(f"{i}. **{result.title}**")
            output.append(f"   URL: {result.url}")
            if result.text:
                snippet = result.text[:500] + "..." if len(result.text) > 500 else result.text
                output.append(f"   {snippet}")
            output.append("")

        return "\n".join(output)

    except Exception as e:
        return f"검색 오류: {str(e)}"


@tool
def search_news(query: str, days: int = 7) -> str:
    """EXA API를 사용하여 최신 뉴스를 검색합니다.

    최근 뉴스, 트렌드, 업계 동향을 파악할 때 사용하세요.

    Args:
        query: 검색할 뉴스 주제
        days: 검색할 기간 (기본값: 최근 7일)
    """
    try:
        start_date = (datetime.now() - timedelta(days=days)).strftime("%Y-%m-%d")

        results = exa_client.search_and_contents(
            query=query,
            type="auto",
            num_results=5,
            start_published_date=start_date,
            text={"max_characters": 2000},
        )

        if not results.results:
            return f"최근 {days}일 내 '{query}' 관련 뉴스가 없습니다."

        output = [f"최근 {days}일 뉴스 ('{query}'):\n"]
        for i, result in enumerate(results.results, 1):
            output.append(f"{i}. **{result.title}**")
            output.append(f"   URL: {result.url}")
            if hasattr(result, 'published_date') and result.published_date:
                output.append(f"   발행일: {result.published_date}")
            if result.text:
                snippet = result.text[:500] + "..." if len(result.text) > 500 else result.text
                output.append(f"   {snippet}")
            output.append("")

        return "\n".join(output)

    except Exception as e:
        return f"뉴스 검색 오류: {str(e)}"


print("✅ EXA 검색 도구 정의 완료!")
print("   - search_web: 웹 검색 (논문, 기사, 문서)")
print("   - search_news: 최신 뉴스 검색 (날짜 필터)")

In [ ]:
import re
from e2b_code_interpreter import Sandbox


@tool
def validate_markdown_section(
    section_name: str,
    markdown: str,
    min_words: int = 50,
    max_words: int = 300,
) -> dict:
    """작성된 섹션의 마크다운 품질을 검증합니다.

    Returns:
        {"ok": bool, "word_count": int, "issues": list[str]}
    """
    issues: list[str] = []
    words = re.findall(r"\b\w+\b", markdown)
    wc = len(words)

    if wc < min_words:
        issues.append(f"너무 짧음: {wc} < {min_words} 단어")
    if wc > max_words:
        issues.append(f"너무 김: {wc} > {max_words} 단어")

    first_lines = "\n".join(markdown.splitlines()[:3]).strip()
    if "##" not in first_lines:
        issues.append("마크다운 제목(예: '## ...')이 상단에 없음")

    return {"ok": len(issues) == 0, "word_count": wc, "issues": issues}


# ── E2B 샌드박스 생성 ──
# timeout: 샌드박스 유지 시간 (초). 기본값 300초(5분)이므로
# 노트북 실습 중 만료되지 않도록 넉넉히 설정합니다.
# 만료 시 셀을 다시 실행하면 새 샌드박스가 생성됩니다.
sandbox = Sandbox.create(timeout=600)  # 10분


@tool
def e2b_code_interpreter(code: str) -> str:
    """E2B 클라우드 샌드박스에서 Python 코드를 실행합니다.

    데이터 분석, 계산, 시각화 등 코드 실행이 필요할 때 사용하세요.
    격리된 환경에서 안전하게 실행되며, matplotlib 등 라이브러리 사용 가능합니다.

    Args:
        code: 실행할 Python 코드
    """
    execution = sandbox.run_code(code)

    result_parts = []
    if execution.text:
        result_parts.append(execution.text)
    if execution.logs.stdout:
        stdout = "".join(execution.logs.stdout)
        if stdout.strip():
            result_parts.append(stdout.strip())
    if execution.logs.stderr:
        stderr = "".join(execution.logs.stderr)
        if stderr.strip():
            result_parts.append(f"[stderr] {stderr.strip()}")
    if execution.error:
        result_parts.append(f"Error: {execution.error.name}: {execution.error.value}")

    return "\n".join(result_parts) if result_parts else "코드가 성공적으로 실행되었습니다."


print("✅ Writer + Code 도구 정의 완료!")
print("   - validate_markdown_section: 마크다운 품질 검증")
print("   - e2b_code_interpreter: E2B 샌드박스 코드 실행")
print(f"   - E2B 샌드박스 생성됨 (timeout: 600초, id: {sandbox.sandbox_id[:8]}...)")

---
## 2. 전문 Sub-agent 생성

3개의 전문 Sub-agent를 `create_agent()`로 생성합니다.

| Sub-agent | 도구 | 역할 |
|-----------|------|------|
| **Research Agent** | `search_web`, `search_news` | EXA로 웹/뉴스 검색 |
| **Writer Agent** | `validate_markdown_section` | 마크다운 섹션 작성 + 검증 |
| **Code Agent** | `e2b_code_interpreter` | E2B 샌드박스에서 코드 실행 |

**핵심 포인트**:
- 각 Sub-agent는 자신만의 **도구**와 **전문 프롬프트**를 가짐
- Sub-agent 내부에서 도구 호출 → 검증 → 수정의 **자율적 루프**가 동작
- Supervisor는 Sub-agent의 내부 동작을 볼 필요 없음 (추상화)

In [ ]:
from datetime import datetime

today = datetime.today().strftime('%Y-%m-%d')
date_prompt=f"""
!!!!! 반드시 주의 !!!!!
[중요] 오늘 날짜는 반드시 {today}입니다. 
절대로 이전 연도(2023/2024 등)가 아닌, {today} 기준으로만 분석하세요.
날짜를 임의로 유추해서 쓰지 말 것.\n"""

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# =====================
# 1) Research Agent: EXA 웹 검색 전문
# =====================

RESEARCH_AGENT_PROMPT = (
    "당신은 기술 리서치 전문 에이전트입니다. "
    "주어진 주제에 대해 search_web으로 관련 논문/기사/문서를 검색하고, "
    "필요하면 search_news로 최신 동향도 파악하세요. "
    "검색 결과를 종합하여 핵심 발견사항을 정리해서 응답하세요. "
    "반드시 출처(URL)와 연도를 포함하세요."
)

research_agent = create_agent(
    llm,
    tools=[search_web, search_news],
    system_prompt=date_prompt+RESEARCH_AGENT_PROMPT,
)

# =====================
# 2) Writer Agent: 섹션 작성 전문
# =====================

WRITER_AGENT_PROMPT = (
    "당신은 기술 보고서 섹션 작성 전문 에이전트입니다. "
    "주어진 리서치 결과를 바탕으로 마크다운 형식의 섹션을 작성하세요. "
    "작성 후 반드시 validate_markdown_section을 호출하여 품질을 검증하세요. "
    "검증에 실패하면 수정 후 다시 검증하세요. "
    "최종 응답에는 완성된 마크다운 섹션만 포함하세요."
)

writer_agent = create_agent(
    llm,
    tools=[validate_markdown_section],
    system_prompt=WRITER_AGENT_PROMPT,
)

# =====================
# 3) Code Agent: E2B 샌드박스 코드 실행 전문
# =====================

CODE_AGENT_PROMPT = (
    "당신은 데이터 분석 및 코드 실행 전문 에이전트입니다. "
    "e2b_code_interpreter를 사용하여 E2B 클라우드 샌드박스에서 Python 코드를 실행하세요. "
    "데이터 분석, 수학 계산, 시각화, 통계 등의 작업을 수행합니다. "
    "코드에서는 print()로 결과를 출력하세요. "
    "에러가 발생하면 코드를 수정하여 다시 실행하세요. "
    "최종 응답에는 분석 결과와 인사이트를 정리해서 포함하세요."
)

code_agent = create_agent(
    llm,
    tools=[e2b_code_interpreter],
    system_prompt=CODE_AGENT_PROMPT,
)

print("✅ 3개 Sub-agent 생성 완료!")
print("   - Research Agent: search_web, search_news (EXA)")
print("   - Writer Agent: validate_markdown_section")
print("   - Code Agent: e2b_code_interpreter (E2B 샌드박스)")

---
### 2-1. Sub-agent 개별 테스트

각 Sub-agent가 자신의 도구를 활용하여 자율적으로 동작하는지 확인합니다.

Sub-agent는 Supervisor와 무관하게 **독립적으로 테스트**할 수 있습니다.
이것이 계층 구조의 장점입니다.

In [ ]:
# Research Agent 테스트 (EXA 실시간 웹 검색)
query = "LLM Scaling Laws에 대한 최신 연구 동향을 조사해주세요"

print(f"📝 Research Agent에 직접 요청: {query}")
print("=" * 60)

for step in research_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

In [ ]:
# Code Agent 테스트 (E2B 샌드박스에서 실행)
query = (
    "1부터 100까지 소수를 구하고, "
    "소수의 분포를 10단위 구간별로 분석해주세요. "
    "결과를 표 형태로 보여주세요."
)

print(f"📝 Code Agent에 직접 요청: {query}")
print("=" * 60)

for step in code_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

---
## 3. Sub-agent를 Tool로 래핑 (핵심 아키텍처 단계)

이것이 Supervisor 패턴의 **핵심 아키텍처 단계**입니다.

각 Sub-agent를 Supervisor가 호출할 수 있는 **고수준 도구(tool)**로 래핑합니다.

```
Supervisor가 보는 것              Sub-agent 내부
───────────────────           ──────────────
research_topic(request)  →  Research Agent가 EXA search_web, search_news 호출
write_section(request)   →  Writer Agent가 validate_markdown_section 호출
run_code(request)        →  Code Agent가 E2B e2b_code_interpreter 호출
```

**중요**: Supervisor는 `search_web` 같은 low-level 도구를 모릅니다.
`research_topic`이라는 고수준 도구만 봅니다.

> 💡 **도구 설명(docstring)**이 Supervisor의 라우팅 결정에 핵심 역할을 합니다.
> 명확하고 구체적으로 작성하세요.

In [ ]:
@tool
def research_topic(request: str) -> str:
    """주제를 리서치하여 핵심 발견사항을 정리합니다.

    기술 동향, 연구 결과, 최신 뉴스가 필요할 때 사용하세요.
    Sub-agent가 EXA API로 웹 검색 및 뉴스 검색을 자동으로 수행합니다.

    Input: 자연어 리서치 요청 (e.g., 'Transformer 아키텍처의 발전 과정 조사')
    """
    result = research_agent.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    # Sub-agent의 최종 응답만 Supervisor에게 반환 (중간 추론/도구 호출은 숨김)
    return result["messages"][-1].content


@tool
def write_section(request: str) -> str:
    """리서치 결과를 바탕으로 보고서 섹션을 작성합니다.

    마크다운 형식의 섹션 작성이 필요할 때 사용하세요.
    Sub-agent가 작성 및 품질 검증을 자동으로 수행합니다.

    Input: 작성할 섹션 정보 (제목, 리서치 결과, 요구사항 포함)
    """
    result = writer_agent.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    return result["messages"][-1].content


@tool
def run_code(request: str) -> str:
    """데이터 분석이나 계산을 위해 Python 코드를 실행합니다.

    수치 분석, 통계 처리, 데이터 시각화가 필요할 때 사용하세요.
    Sub-agent가 E2B 샌드박스에서 안전하게 코드를 작성하고 실행합니다.

    Input: 자연어 분석 요청 (e.g., '연도별 논문 수 추이를 분석하고 차트 생성')
    """
    result = code_agent.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    return result["messages"][-1].content


print("✅ Sub-agent → Tool 래핑 완료!")
print("   - research_topic → Research Agent (EXA 웹 검색)")
print("   - write_section  → Writer Agent (마크다운 작성 + 검증)")
print("   - run_code       → Code Agent (E2B 샌드박스 실행)")

---
## 4. Supervisor Agent 생성

이제 **Supervisor**를 생성합니다.

Supervisor는 고수준 도구(`research_topic`, `write_section`, `run_code`)만 보고,
사용자 요청을 분석하여 적절한 Sub-agent에게 작업을 위임합니다.

복합 요청의 경우 여러 도구를 **순차적 또는 병렬로** 호출할 수 있습니다.

In [ ]:
SUPERVISOR_PROMPT = (
    "당신은 기술 보고서 작성을 총괄하는 Supervisor 에이전트입니다. "
    "사용자의 요청을 분석하여 적절한 도구를 호출하세요:\n\n"
    "1. research_topic: 주제 리서치가 필요할 때 (EXA 웹 검색)\n"
    "2. write_section: 리서치 결과를 바탕으로 섹션 작성할 때\n"
    "3. run_code: 데이터 분석, 계산, 시각화가 필요할 때 (E2B 샌드박스)\n\n"
    "복합 요청의 경우, 먼저 리서치하고 필요시 코드 분석을 한 후, "
    "그 결과를 바탕으로 섹션을 작성하세요. "
    "최종 응답에서 전체 결과를 종합하여 사용자에게 보고하세요."
)

supervisor = create_agent(
    llm,
    tools=[research_topic, write_section, run_code],
    system_prompt=SUPERVISOR_PROMPT,
)

print("✅ Supervisor Agent 생성 완료!")

---
## 5. Supervisor 실행 및 테스트

### 테스트 1: 단일 도메인 요청 (리서치만)

Supervisor가 요청을 분석하여 적절한 Sub-agent에게 위임하는지 확인합니다.

In [ ]:
query = "최근 LLM Agent 프레임워크 동향에 대해 조사해주세요"

print(f"💬 사용자 요청: {query}")
print("=" * 60)

for step in supervisor.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

### 테스트 2: 복합 요청 (리서치 → 분석 → 작성)

Supervisor가 **3개 Sub-agent를 순차적으로 조율**하는지 확인합니다.

1. Research Agent: 웹에서 정보 수집
2. Code Agent: 수집된 데이터 분석
3. Writer Agent: 결과를 보고서 섹션으로 작성

In [ ]:
query = (
    "LLM Scaling Laws에 대해 웹에서 리서치한 후, "
    "2025년과 2026년에 발표된 주요 모델들의 파라미터 수나 다양한 벤치마크 성적 등을 코드로 비교 분석하고, "
    "그 결과를 바탕으로 '2025년 이후 Scaling Laws의 현황' 보고서 섹션을 작성해주세요."
)

print(f"💬 사용자 요청: {query}")
print("=" * 60)

for step in supervisor.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

---
## 6. (Advanced) ToolRuntime으로 Supervisor 컨텍스트 전달

기본적으로 Sub-agent는 Supervisor가 전달한 `request` 문자열만 받습니다.

`ToolRuntime`을 사용하면 **Supervisor의 전체 대화 맥락**을 Sub-agent에게 전달할 수 있습니다.

예: 사용자가 "대상 독자는 AI 엔지니어" 같은 맥락을 제공한 경우,
Sub-agent가 이 맥락을 참조하여 더 적절한 결과를 생성합니다.

```python
@tool
def my_tool(request: str, runtime: ToolRuntime) -> str:
    # runtime.state["messages"] → Supervisor의 전체 대화 기록 접근 가능
    original_user_msg = next(
        m for m in runtime.state["messages"] if m.type == "human"
    )
```

In [ ]:
from langchain.tools import ToolRuntime


@tool
def research_topic_with_context(
    request: str,
    runtime: ToolRuntime,
) -> str:
    """Supervisor의 전체 대화 맥락을 포함하여 리서치합니다."""
    original_user_message = next(
        (m for m in runtime.state["messages"] if m.type == "human"),
        None,
    )
    supervisor_context = (
        original_user_message.content if original_user_message else ""
    )

    enriched_prompt = (
        f"[Supervisor 전체 맥락]\n{supervisor_context}\n\n"
        f"[현재 리서치 요청]\n{request}"
    )

    result = research_agent.invoke(
        {"messages": [{"role": "user", "content": enriched_prompt}]}
    )
    return result["messages"][-1].content


@tool
def write_section_with_context(
    request: str,
    runtime: ToolRuntime,
) -> str:
    """Supervisor의 전체 대화 맥락을 포함하여 섹션을 작성합니다."""
    original_user_message = next(
        (m for m in runtime.state["messages"] if m.type == "human"),
        None,
    )
    supervisor_context = (
        original_user_message.content if original_user_message else ""
    )

    enriched_prompt = (
        f"[Supervisor 전체 맥락]\n{supervisor_context}\n\n"
        f"[현재 작성 요청]\n{request}"
    )

    result = writer_agent.invoke(
        {"messages": [{"role": "user", "content": enriched_prompt}]}
    )
    return result["messages"][-1].content


@tool
def run_code_with_context(
    request: str,
    runtime: ToolRuntime,
) -> str:
    """Supervisor의 전체 대화 맥락을 포함하여 코드를 실행합니다."""
    original_user_message = next(
        (m for m in runtime.state["messages"] if m.type == "human"),
        None,
    )
    supervisor_context = (
        original_user_message.content if original_user_message else ""
    )

    enriched_prompt = (
        f"[Supervisor 전체 맥락]\n{supervisor_context}\n\n"
        f"[현재 분석 요청]\n{request}"
    )

    result = code_agent.invoke(
        {"messages": [{"role": "user", "content": enriched_prompt}]}
    )
    return result["messages"][-1].content


# ToolRuntime 버전 Supervisor
supervisor_with_context = create_agent(
    llm,
    tools=[
        research_topic_with_context,
        write_section_with_context,
        run_code_with_context,
    ],
    system_prompt=SUPERVISOR_PROMPT,
)

print("✅ ToolRuntime 버전 Supervisor 생성 완료!")
print("   Sub-agent가 Supervisor의 전체 대화 맥락을 참조할 수 있습니다.")

In [ ]:
# ToolRuntime 테스트: "대상 독자", "관점" 같은 맥락이 Sub-agent에 전달됨
query = (
    "LLM Scaling Laws에 대한 기술 보고서를 작성하려고 합니다. "
    "대상 독자는 AI 엔지니어이고, 실무 적용 관점을 강조해주세요. "
    "먼저 Scaling Laws에 대해 웹에서 조사한 후, '핵심 원리' 섹션을 작성해주세요."
)

print(f"💬 사용자 요청: {query}")
print("=" * 60)

for step in supervisor_with_context.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

---
## 7. (Advanced) Human-in-the-loop 리뷰

중요한 작업(외부 API 호출, 코드 실행 등)에 **사람의 승인**을 추가할 수 있습니다.

`HumanInTheLoopMiddleware`를 사용하면:
- 특정 도구 호출 전에 **실행을 일시 중지** (interrupt)
- 사용자가 **승인(approve)**, **수정(edit)**, **거부(reject)** 가능
- 승인 후 실행 재개

> ⚠️ `checkpointer`가 **최상위 에이전트(Supervisor)에만** 필요합니다.
> 실행 일시중지/재개를 위한 상태 저장에 사용됩니다.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# Code Agent에 HITL 추가 - 코드 실행 전에 사람이 검토
code_agent_with_review = create_agent(
    llm,
    tools=[e2b_code_interpreter],
    system_prompt=CODE_AGENT_PROMPT,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"e2b_code_interpreter": True},
            description_prefix="코드 실행 전 사람의 검토 필요",
        ),
    ],
)


@tool
def run_code_reviewed(request: str) -> str:
    """사람의 검토를 거쳐 코드를 실행합니다."""
    result = code_agent_with_review.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    return result["messages"][-1].content


# Supervisor (checkpointer 필수 - 실행 일시중지/재개에 필요)
supervisor_with_hitl = create_agent(
    llm,
    tools=[research_topic, write_section, run_code_reviewed],
    system_prompt=SUPERVISOR_PROMPT,
    checkpointer=InMemorySaver(),
)

print("✅ Human-in-the-loop Supervisor 생성 완료!")

In [ ]:
query = "GPT-4와 LLaMA 3의 파라미터 수를 비교하는 코드를 실행해주세요"
config = {"configurable": {"thread_id": "hitl-demo-1"}}

print(f"💬 사용자 요청: {query}")
print("=" * 60)

interrupts = []
for step in supervisor_with_hitl.stream(
    {"messages": [{"role": "user", "content": query}]},
    config,
):
    for update in step.values():
        if isinstance(update, dict):
            for message in update.get("messages", []):
                message.pretty_print()
        else:
            interrupt_ = update[0]
            interrupts.append(interrupt_)
            print(f"\n⏸️ INTERRUPTED: {interrupt_.id}")
            for req in interrupt_.value.get("action_requests", []):
                print(f"   {req.get('description', '')[:200]}")

In [ ]:
from langgraph.types import Command

if interrupts:
    # 모든 인터럽트 승인 (실제로는 사용자가 확인/수정/거부 결정)
    resume = {}
    for interrupt_ in interrupts:
        resume[interrupt_.id] = {"decisions": [{"type": "approve"}]}

    print("✅ 모든 작업 승인 - 실행 재개")
    print("=" * 60)

    for step in supervisor_with_hitl.stream(
        Command(resume=resume),
        config,
    ):
        for update in step.values():
            if isinstance(update, dict):
                for message in update.get("messages", []):
                    message.pretty_print()
            else:
                interrupt_ = update[0]
                interrupts.append(interrupt_)
                print(f"\n⏸️ INTERRUPTED: {interrupt_.id}")
else:
    print("인터럽트 없이 실행 완료!")

---
## 8. 정리 및 리소스 해제

In [ ]:
# E2B 샌드박스 종료 (비용 절약)
sandbox.kill()
print("✅ E2B 샌드박스가 종료되었습니다.")

---
## 💡 정리

### 핵심 개념 정리

| 구성요소 | 역할 | 핵심 기술 |
|---------|------|----------|
| **Low-level 도구** | 실제 API 호출 | EXA (웹 검색), E2B (코드 실행) |
| **Research Agent** | 웹/뉴스 검색 전문 | `create_agent()` + EXA `search_web`, `search_news` |
| **Writer Agent** | 보고서 섹션 작성 | `create_agent()` + `validate_markdown_section` |
| **Code Agent** | 데이터 분석/코드 실행 | `create_agent()` + E2B `e2b_code_interpreter` |
| **Tool 래핑** | Sub-agent → Supervisor 도구 | Sub-agent를 `@tool` 함수로 래핑 |
| **Supervisor** | 작업 분석 + 위임 + 결과 종합 | `create_agent()` + 고수준 도구 3개 |
| **ToolRuntime** | 맥락 전달 | `runtime.state`로 Supervisor 대화 참조 |
| **HITL** | 사람의 검토/승인 | `HumanInTheLoopMiddleware` |

### 3단 계층 아키텍처의 장점

- **관심사 분리**: 각 계층이 명확한 책임을 가짐
- **독립적 개선**: Sub-agent를 개별적으로 테스트하고 개선 가능
- **확장 용이**: 새로운 도메인(예: 차트 생성 Agent)을 기존 코드 변경 없이 추가
- **동적 위임**: Supervisor가 상황에 따라 도구를 선택적으로 호출

### 사용된 외부 서비스

| 서비스 | 용도 | 링크 |
|--------|------|------|
| **EXA** | AI-native 웹 검색 API | https://exa.ai |
| **E2B** | 클라우드 샌드박스 코드 실행 | https://e2b.dev |

### 다음 단계
👉 **CH02._06.** Evaluator-Optimizer으로 자율 개선 만들기